# Week 5 – ML Frameworks for Chooser Option Pricing

This notebook demonstrates the initial ML pipelines for:

- **Approach 1:** Volatility forecasting (RF/XGBoost) + BSM pricing.
- **Approach 2:** End-to-end supervised pricing (Linear / GBDT / MLP).

The goal is to wire up datasets, time-series splits, and basic models; full tuning and comparison will follow in later weeks.

In [ ]:
import sys
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ml.datasets import (
    load_base_frame,
    build_volatility_dataset,
    build_pricing_dataset,
    time_series_split,
)
from src.ml.models_vol import train_rf_vol
from src.ml.models_pricing import train_linear_pricing
from src.ml.metrics import regression_metrics

print("Project root:", PROJECT_ROOT)

In [ ]:
# Load base frame and build datasets
frame = load_base_frame("JPM")
print("Base frame shape:", frame.shape)

# Example: volatility dataset with 126-day horizon
X_vol, y_vol, dates_vol = build_volatility_dataset(frame, horizon_days=126)
(train_vol, val_vol, test_vol) = time_series_split(X_vol, y_vol, dates_vol)

print("Volatility dataset:")
print("  X shape:", X_vol.shape)
print("  y shape:", y_vol.shape)

In [ ]:
# Quick RF volatility model demo (train vs val metrics)
X_train_v, y_train_v, _ = train_vol[0], train_vol[1], train_vol[2]
X_val_v, y_val_v, _ = val_vol[0], val_vol[1], val_vol[2]

rf_vol_model, rf_vol_metrics = train_rf_vol(X_train_v, y_train_v, X_val_v, y_val_v)
print("Random Forest vol metrics (val):", rf_vol_metrics)

In [ ]:
# Example: pricing dataset (using config-like defaults)
K = 150.0
R_DEFAULT = 0.0015
Q = 0.0233
T1_YEARS = 0.5
T2_YEARS = 1.0
T1_DAYS = 126
T2_DAYS = 252

X_price, y_price, dates_price, bsm_prices = build_pricing_dataset(
    frame,
    k=K,
    r_default=R_DEFAULT,
    q=Q,
    t1_years=T1_YEARS,
    t2_years=T2_YEARS,
    t1_days=T1_DAYS,
    t2_days=T2_DAYS,
)

(train_price, val_price, test_price) = time_series_split(X_price, y_price, dates_price)

print("Pricing dataset:")
print("  X shape:", X_price.shape)
print("  y shape:", y_price.shape)

In [ ]:
# Quick linear pricing model demo (on proxy-actual target)
X_train_p, y_train_p, _ = train_price[0], train_price[1], train_price[2]
X_val_p, y_val_p, _ = val_price[0], val_price[1], val_price[2]

lin_model, lin_metrics = train_linear_pricing(X_train_p, y_train_p, X_val_p, y_val_p)
print("Linear pricing metrics (val):", lin_metrics)